# 02 — Feature Engineering: Weather Join + Calendar Features

**Project:** Chicago Transit & Logistics Intelligence Platform  
**Author:** Hari Etta  
**Purpose:** Build the full feature set used by the Prophet forecasting model and DiD analysis.

## What this notebook covers
1. Load ridership data from EDA output
2. Fetch and join historical weather (Open-Meteo)
3. Build calendar features — day of week, week of year, holidays
4. Build weather features — temperature buckets, precipitation flags, impact score
5. Validate feature quality — null rates, distributions, correlations
6. Feature importance — which features most predict ridership?
7. Save feature-engineered dataset for forecasting notebook

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv('../.env')

import sys
sys.path.insert(0, '..')
from ingestion.weather_api_client import OpenMeteoClient

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Libraries loaded.')

## 1. Load Ridership Data

In [ ]:
try:
    df = pd.read_parquet('../data/processed/eda_ridership.parquet')
    print(f'Loaded {len(df):,} rows from processed parquet.')
except FileNotFoundError:
    df = pd.read_csv('../data/sample/cta_ridership_sample.csv', parse_dates=['service_date'])
    print(f'Loaded {len(df):,} rows from sample CSV.')

df['service_date'] = pd.to_datetime(df['service_date'])
df['rides'] = pd.to_numeric(df['rides'], errors='coerce')
print(f'Date range: {df["service_date"].min().date()} → {df["service_date"].max().date()}')
df.head(3)

## 2. Fetch and Join Weather Data

In [ ]:
# Only fetch if weather columns are missing or mostly null
weather_needs_fetch = (
    'temperature_2m' not in df.columns
    or df['temperature_2m'].isnull().mean() > 0.5
)

if weather_needs_fetch:
    print('Fetching weather data from Open-Meteo...')
    client = OpenMeteoClient()
    start = df['service_date'].min().date()
    end = df['service_date'].max().date()
    weather_records = client.fetch_historical_weather(start, end)
    weather_df = pd.DataFrame(weather_records)
    weather_df['date'] = pd.to_datetime(weather_df['date'])

    # Use 8 AM as daily representative hour for commute context
    morning_weather = weather_df[weather_df['hour'] == 8].copy()
    morning_weather = morning_weather.rename(columns={'date': 'service_date'})

    # Daily totals for precipitation
    daily_precip = (
        weather_df.groupby('date')['precipitation']
        .sum()
        .reset_index()
        .rename(columns={'date': 'service_date', 'precipitation': 'daily_precipitation'})
    )
    daily_precip['service_date'] = pd.to_datetime(daily_precip['service_date'])

    # Drop existing weather cols and rejoin
    drop_cols = [c for c in ['temperature_2m','precipitation','windspeed_10m','weathercode','is_precipitation'] if c in df.columns]
    df = df.drop(columns=drop_cols)
    df = df.merge(
        morning_weather[['service_date','temperature_2m','apparent_temperature','windspeed_10m','weathercode','is_precipitation']],
        on='service_date', how='left'
    )
    df = df.merge(daily_precip, on='service_date', how='left')
    df['precipitation'] = df['daily_precipitation']
    df = df.drop(columns=['daily_precipitation'])
    print(f'Weather joined: {df["temperature_2m"].notna().sum():,} rows with temperature data.')
else:
    print('Weather columns already present — skipping fetch.')
    df['apparent_temperature'] = df.get('apparent_temperature', np.nan)

df.head(3)

## 3. Calendar Features

In [ ]:
# Chicago public holidays
years = df['service_date'].dt.year.unique().tolist()
chicago_holidays = set()
for y in years:
    chicago_holidays.update(hol.US(years=y, state='IL').keys())

df['is_holiday'] = df['service_date'].dt.date.isin(chicago_holidays)

# Calendar features
df['day_of_week'] = df['service_date'].dt.dayofweek          # 0=Mon, 6=Sun
df['day_name'] = df['service_date'].dt.day_name()
df['month'] = df['service_date'].dt.month
df['month_name'] = df['service_date'].dt.month_name()
df['week_of_year'] = df['service_date'].dt.isocalendar().week.astype(int)
df['year'] = df['service_date'].dt.year
df['year_month'] = df['service_date'].dt.to_period('M').astype(str)
df['quarter'] = df['service_date'].dt.quarter
df['is_weekday'] = df['day_type'] == 'W'
df['is_weekend'] = df['day_type'].isin(['A', 'U'])
df['is_month_start'] = df['service_date'].dt.is_month_start
df['is_month_end'] = df['service_date'].dt.is_month_end

# Cyclical encoding for day_of_week and month (preserves circular distance)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print('Calendar features added:')
cal_cols = ['day_of_week','month','week_of_year','is_holiday','is_weekday','dow_sin','month_sin']
print(df[cal_cols].head(5))

In [ ]:
# Holiday impact analysis
holiday_impact = (
    df.groupby('is_holiday')['rides']
    .agg(['mean','median','count'])
    .round(0)
)
holiday_impact.index = ['Non-holiday','Holiday']
holiday_impact['pct_change'] = (
    (holiday_impact['mean'] - holiday_impact.loc['Non-holiday','mean'])
    / holiday_impact.loc['Non-holiday','mean'] * 100
).round(1)
print('Holiday impact on ridership:')
print(holiday_impact)

## 4. Weather Features

In [ ]:
# Temperature buckets
def temp_bucket(t):
    if pd.isna(t): return 'unknown'
    if t < -10: return 'extreme_cold'
    if t < 0:   return 'cold'
    if t < 10:  return 'cool'
    if t < 20:  return 'mild'
    if t < 30:  return 'warm'
    return 'hot'

df['temp_bucket'] = df['temperature_2m'].apply(temp_bucket)
df['is_heavy_precip'] = df['precipitation'].fillna(0) > 5.0
df['is_precipitation'] = df['precipitation'].fillna(0) > 0.1

# Weather impact score (0-100)
def weather_impact(row):
    score = 0
    if row.get('is_heavy_precip', False):             score += 35
    if row.get('temperature_2m', 15) < -10:           score += 25
    elif row.get('temperature_2m', 15) < 0:           score += 10
    if row.get('windspeed_10m', 0) > 50:              score += 20
    elif row.get('windspeed_10m', 0) > 30:            score += 8
    if row.get('is_precipitation', False) and not row.get('is_heavy_precip', False):
        score += 10
    return min(100, score)

df['weather_impact_score'] = df.apply(weather_impact, axis=1)

print('Weather feature distribution:')
print(df[['temp_bucket','is_precipitation','is_heavy_precip','weather_impact_score']].describe(include='all'))

In [ ]:
# Weather impact score vs ridership
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Impact score distribution
axes[0].hist(df['weather_impact_score'], bins=20, color='coral', edgecolor='white')
axes[0].set_title('Weather Impact Score Distribution', fontweight='bold')
axes[0].set_xlabel('Weather Impact Score (0=clear, 100=severe)')
axes[0].set_ylabel('Days')

# Score vs ridership (weekdays only, Route 22)
r22_w = df[(df['route'] == '22') & (df['is_weekday'])].dropna(subset=['weather_impact_score','rides'])
axes[1].scatter(r22_w['weather_impact_score'], r22_w['rides'], alpha=0.3, s=10, color='steelblue')
axes[1].set_title('Route 22: Weather Impact Score vs Daily Rides', fontweight='bold')
axes[1].set_xlabel('Weather Impact Score')
axes[1].set_ylabel('Daily Rides')
corr = r22_w[['weather_impact_score','rides']].corr().iloc[0,1]
axes[1].set_title(f'Route 22: Weather Impact Score vs Daily Rides\nPearson r = {corr:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/feature_weather_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance (Random Forest proxy)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

feature_cols = [
    'day_of_week', 'month', 'week_of_year', 'is_holiday',
    'is_weekday', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'temperature_2m', 'precipitation', 'windspeed_10m', 'weather_impact_score'
]

r22_model = df[df['route'] == '22'].copy()
r22_model = r22_model.dropna(subset=feature_cols + ['rides'])

X = r22_model[feature_cols].astype(float)
y = r22_model['rides']

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance for Route 22 Ridership (Random Forest)', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features:')
print(importance.tail(5).round(4))

## 6. Final Feature Set Validation

In [ ]:
final_feature_cols = [
    'route', 'service_date', 'rides', 'day_type',
    # Calendar
    'day_of_week', 'day_name', 'month', 'month_name',
    'week_of_year', 'year', 'year_month', 'quarter',
    'is_weekday', 'is_weekend', 'is_holiday',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    # Weather
    'temperature_2m', 'apparent_temperature', 'precipitation',
    'windspeed_10m', 'weathercode', 'is_precipitation',
    'is_heavy_precip', 'temp_bucket', 'weather_impact_score'
]

df_features = df[final_feature_cols].copy()

print('=== Final Feature Set ===')
print(f'Shape: {df_features.shape}')
print(f'\nNull rates:')
null_rates = (df_features.isnull().mean() * 100).round(1)
print(null_rates[null_rates > 0].to_string() if (null_rates > 0).any() else 'No nulls in core columns.')

# Save
df_features.to_parquet('../data/processed/features.parquet', index=False)
print(f'\nSaved feature dataset: {len(df_features):,} rows → data/processed/features.parquet')